In [ ]:
#Jawahar babu S
#212224220041


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Model(nn.Module):
    
    def __init__(self, in_features=4, h1=8, h2=9, out_features=3):
        # Input Layers (4 features) ---> h1 --> h2 ---> output (3 classes)
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)   # input layer
        self.fc2 = nn.Linear(h1, h2)            # hidden layer
        self.out = nn.Linear(h2, out_features)  # output layer
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

In [ ]:
# Instantiate the Model class using parameter defaults:
torch.manual_seed(32)
model = Model()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# FIX: Load iris dataset directly from sklearn (no CSV file needed)
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame
df.head()

In [ ]:
df.tail()

In [ ]:
# Features and labels
X = df.drop('target', axis=1).values
y = df['target'].values

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

# Convert to tensors
X_train = torch.FloatTensor(X_train)
X_test  = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test  = torch.LongTensor(y_test)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Train the model
epochs = 100
losses = []

for i in range(epochs):
    y_pred = model.forward(X_train)
    loss = criterion(y_pred, y_train)
    losses.append(loss.detach().numpy())

    if i % 10 == 0:
        print(f'Epoch {i} Loss: {loss.item():.4f}')

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
plt.plot(range(epochs), losses)
plt.ylabel('Loss')
plt.xlabel('Epoch');

In [ ]:
correct = 0

with torch.no_grad():
    for i, data in enumerate(X_test):
        y_val = model.forward(data)
        print(f'{i+1}.) {str(y_val)} {y_test[i]}')
        if y_val.argmax().item() == y_test[i]:
            correct += 1

print(f'We got {correct} correct!')

In [ ]:
print(f'{i+1}.) {str(y_val.argmax().item())} {y_test[i]}')

In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'my_iris_model.pt')
print('Model saved as my_iris_model.pt')

In [ ]:
# Reload and evaluate the saved model
# FIX: Use the correct architecture (matching the saved model: h1=8, h2=9)
# FIX: Load from correct filename 'my_iris_model.pt'

new_model = Model()  # same architecture as original
new_model.load_state_dict(torch.load('my_iris_model.pt'))
new_model.eval()
print('Model loaded successfully!')

In [ ]:
# Test new_model on a sample
sample = X_test[0].unsqueeze(0)
with torch.no_grad():
    out = new_model(sample)
    pred = out.argmax().item()
print(f'Predicted class: {pred}, Actual class: {y_test[0].item()}')